<a href="https://colab.research.google.com/github/ferhat00/LLM/blob/main/fpl_keras_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FPL Squad Optimiser with Keras Neural Networks

**Features:**
- Uses **3 seasons of data** (current + 2 previous) for training
- **Keras Deep Neural Network** for point prediction
- **Keras Tuner** for automatic hyperparameter optimization
- Transfer recommendations with user-specified number of transfers
- Respects all FPL constraints

**Data Sources:**
- Current season: Official FPL API
- Historical seasons: vaastav/Fantasy-Premier-League GitHub repository

**Updated:** 2025

In [1]:
# Install required packages
!pip install pulp keras-tuner tensorflow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.3 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import json
import requests
import warnings
import os
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import keras_tuner as kt
import pulp

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ Keras Tuner version: {kt.__version__}")
print(f"✅ All packages loaded successfully!")

✅ TensorFlow version: 2.19.0
✅ Keras Tuner version: 1.4.8
✅ All packages loaded successfully!


In [3]:
# ----------------------------------------------------------------------
# 1. DATA FETCHING - CURRENT SEASON + 2 HISTORICAL SEASONS
# ----------------------------------------------------------------------

# Historical data source (vaastav's excellent FPL data repository)
GITHUB_BASE = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data"

# Seasons to use (current + 2 previous)
SEASONS = ['2024-25', '2023-24', '2022-23']


def fetch_current_season_data(verify_ssl: bool = True) -> pd.DataFrame:
    """
    Fetch current season data from the official FPL API.
    """
    print("📡 Fetching current season (2024-25) from FPL API...")

    if not verify_ssl:
        import urllib3
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = "https://fantasy.premierleague.com/api/bootstrap-static/"

    try:
        response = requests.get(url, timeout=30, verify=verify_ssl)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        print(f"❌ Failed to fetch current season: {e}")
        return pd.DataFrame()

    teams = {team['id']: team['name'] for team in data['teams']}
    positions = {1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'}

    players = []
    for player in data['elements']:
        pos = positions[player['element_type']]

        players.append({
            'player_id': player['id'],
            'name': player['web_name'],
            'full_name': f"{player['first_name']} {player['second_name']}",
            'team': teams[player['team']],
            'position': pos,
            'price': player['now_cost'] / 10.0,
            'total_points': player['total_points'],
            'minutes': player['minutes'],
            'goals_scored': player['goals_scored'],
            'assists': player['assists'],
            'clean_sheets': player['clean_sheets'],
            'goals_conceded': player['goals_conceded'],
            'own_goals': player['own_goals'],
            'penalties_saved': player['penalties_saved'],
            'penalties_missed': player['penalties_missed'],
            'yellow_cards': player['yellow_cards'],
            'red_cards': player['red_cards'],
            'saves': player['saves'],
            'bonus': player['bonus'],
            'bps': player['bps'],
            'influence': float(player['influence']),
            'creativity': float(player['creativity']),
            'threat': float(player['threat']),
            'ict_index': float(player['ict_index']),
            'starts': player['starts'],
            'expected_goals': float(player.get('expected_goals', 0) or 0),
            'expected_assists': float(player.get('expected_assists', 0) or 0),
            'expected_goal_involvements': float(player.get('expected_goal_involvements', 0) or 0),
            'expected_goals_conceded': float(player.get('expected_goals_conceded', 0) or 0),
            'form': float(player['form']) if player['form'] else 0,
            'selected_by_percent': float(player['selected_by_percent']),
            'status': player['status'],
            'season': '2024-25',
            'is_current_season': True
        })

    df = pd.DataFrame(players)
    print(f"   ✅ Loaded {len(df)} players from current season")
    return df


def fetch_historical_season(season: str) -> pd.DataFrame:
    """
    Fetch historical season data from vaastav's GitHub repository.
    """
    print(f"📡 Fetching historical season ({season}) from GitHub...")

    # Try different file names used in the repo
    urls_to_try = [
        f"{GITHUB_BASE}/{season}/players_raw.csv",
        f"{GITHUB_BASE}/{season}/cleaned_players.csv",
    ]

    df = None
    for url in urls_to_try:
        try:
            df = pd.read_csv(url)
            print(f"   ✅ Loaded {len(df)} players from {season}")
            break
        except Exception as e:
            continue

    if df is None:
        print(f"   ⚠️ Could not fetch {season} data, skipping...")
        return pd.DataFrame()

    # Standardize column names
    column_mapping = {
        'id': 'player_id',
        'web_name': 'name',
        'first_name': 'first_name',
        'second_name': 'second_name',
        'team': 'team_id',
        'element_type': 'position_id',
        'now_cost': 'price',
    }

    df = df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns})

    # Convert position ID to position name if needed
    if 'position_id' in df.columns:
        pos_map = {1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'}
        df['position'] = df['position_id'].map(pos_map)

    # Normalize price (stored as integer * 10 in raw data)
    if 'price' in df.columns and df['price'].max() > 50:
        df['price'] = df['price'] / 10.0

    df['season'] = season
    df['is_current_season'] = False

    return df


def fetch_all_seasons_data(verify_ssl: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fetch data from current season + 2 historical seasons.

    Returns:
        training_df: Combined data from all seasons for model training
        current_df: Current season data for predictions/transfers
    """
    print("\n" + "=" * 70)
    print("📊 FETCHING MULTI-SEASON FPL DATA")
    print("=" * 70 + "\n")

    all_dfs = []

    # Current season
    current_df = fetch_current_season_data(verify_ssl=verify_ssl)
    if not current_df.empty:
        all_dfs.append(current_df)

    # Historical seasons
    for season in SEASONS[1:]:  # Skip current season (index 0)
        hist_df = fetch_historical_season(season)
        if not hist_df.empty:
            all_dfs.append(hist_df)

    if not all_dfs:
        raise RuntimeError("Failed to fetch any season data!")

    # Combine all seasons
    training_df = pd.concat(all_dfs, ignore_index=True)

    print(f"\n📈 Total dataset: {len(training_df)} player-seasons")
    print(f"   Seasons loaded: {training_df['season'].nunique()}")
    for season in training_df['season'].unique():
        count = len(training_df[training_df['season'] == season])
        print(f"   • {season}: {count} players")

    return training_df, current_df

In [4]:
# ----------------------------------------------------------------------
# 2. FEATURE ENGINEERING
# ----------------------------------------------------------------------

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create advanced features for the neural network.
    """
    df = df.copy()

    # Per 90 minute stats (avoid division by zero)
    df['minutes_safe'] = df['minutes'].clip(lower=1)
    df['per_90_factor'] = 90 / df['minutes_safe']

    df['goals_per_90'] = df['goals_scored'] * df['per_90_factor']
    df['assists_per_90'] = df['assists'] * df['per_90_factor']
    df['cs_per_90'] = df['clean_sheets'] * df['per_90_factor']
    df['bonus_per_90'] = df['bonus'] * df['per_90_factor']
    df['bps_per_90'] = df['bps'] * df['per_90_factor']
    df['points_per_90'] = df['total_points'] * df['per_90_factor']

    # Expected stats per 90
    if 'expected_goals' in df.columns:
        df['xg_per_90'] = df['expected_goals'] * df['per_90_factor']
        df['xa_per_90'] = df['expected_assists'] * df['per_90_factor']
        df['xgi_per_90'] = df['expected_goal_involvements'] * df['per_90_factor']

    # ICT components per 90
    df['influence_per_90'] = df['influence'] * df['per_90_factor']
    df['creativity_per_90'] = df['creativity'] * df['per_90_factor']
    df['threat_per_90'] = df['threat'] * df['per_90_factor']

    # Goal involvement
    df['goal_involvements'] = df['goals_scored'] + df['assists']
    df['gi_per_90'] = df['goal_involvements'] * df['per_90_factor']

    # Points per million (value)
    df['price_safe'] = df['price'].clip(lower=4.0)
    df['points_per_million'] = df['total_points'] / df['price_safe']

    # Minutes played ratio (assuming ~38 games * 90 mins = 3420 max)
    df['minutes_pct'] = df['minutes'] / 3420 * 100

    # Starts ratio
    if 'starts' in df.columns:
        df['start_ratio'] = df['starts'] / 38

    # Penalty stats
    if 'penalties_saved' in df.columns:
        df['penalty_involvement'] = df.get('penalties_saved', 0) + df.get('penalties_missed', 0)

    # Defensive stats per 90
    if 'goals_conceded' in df.columns:
        df['goals_conceded_per_90'] = df['goals_conceded'] * df['per_90_factor']

    if 'saves' in df.columns:
        df['saves_per_90'] = df['saves'] * df['per_90_factor']

    # Card risk
    df['cards_per_90'] = (df['yellow_cards'] + df['red_cards'] * 2) * df['per_90_factor']

    # Clean up infinities
    df = df.replace([np.inf, -np.inf], 0)

    return df


def prepare_nn_features(df: pd.DataFrame, fit_encoders: bool = True,
                        scaler: StandardScaler = None,
                        position_encoder: LabelEncoder = None) -> Tuple:
    """
    Prepare features for neural network training.
    """
    # Feature columns for NN
    numeric_features = [
        'price', 'minutes', 'minutes_pct',
        'goals_scored', 'assists', 'clean_sheets', 'bonus', 'bps',
        'goals_per_90', 'assists_per_90', 'cs_per_90', 'bonus_per_90', 'bps_per_90',
        'influence', 'creativity', 'threat', 'ict_index',
        'influence_per_90', 'creativity_per_90', 'threat_per_90',
        'goal_involvements', 'gi_per_90', 'points_per_million',
        'cards_per_90'
    ]

    # Add xG features if available
    xg_features = ['expected_goals', 'expected_assists', 'expected_goal_involvements',
                   'xg_per_90', 'xa_per_90', 'xgi_per_90']
    for feat in xg_features:
        if feat in df.columns:
            numeric_features.append(feat)

    # Filter to existing columns
    numeric_features = [f for f in numeric_features if f in df.columns]

    X_numeric = df[numeric_features].fillna(0).values

    # Encode position
    if fit_encoders:
        position_encoder = LabelEncoder()
        position_encoded = position_encoder.fit_transform(df['position'])
    else:
        position_encoded = position_encoder.transform(df['position'])

    # One-hot encode position
    position_onehot = np.zeros((len(df), 4))
    position_onehot[np.arange(len(df)), position_encoded] = 1

    # Combine features
    X = np.hstack([X_numeric, position_onehot])

    # Scale features
    if fit_encoders:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = scaler.transform(X)

    # Target: points per 90 (more stable than raw points)
    y = df['points_per_90'].fillna(0).values

    return X_scaled, y, scaler, position_encoder, numeric_features

In [5]:
# ----------------------------------------------------------------------
# 3. KERAS NEURAL NETWORK WITH HYPERPARAMETER TUNING
# ----------------------------------------------------------------------

def build_model(hp, input_dim: int):
    """
    Build a Keras model with hyperparameters from Keras Tuner.
    """
    model = keras.Sequential()

    # Input layer
    model.add(layers.Input(shape=(input_dim,)))

    # Number of hidden layers
    n_layers = hp.Int('n_layers', min_value=2, max_value=5, default=3)

    for i in range(n_layers):
        # Units for this layer
        units = hp.Int(f'units_{i}', min_value=32, max_value=256, step=32, default=128)

        # Regularization
        l2_reg = hp.Float(f'l2_{i}', min_value=1e-6, max_value=1e-2, sampling='log', default=1e-4)

        model.add(layers.Dense(
            units=units,
            activation='relu',
            kernel_regularizer=regularizers.l2(l2_reg)
        ))

        # Batch normalization
        if hp.Boolean(f'batch_norm_{i}', default=True):
            model.add(layers.BatchNormalization())

        # Dropout
        dropout_rate = hp.Float(f'dropout_{i}', min_value=0.0, max_value=0.5, step=0.1, default=0.2)
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    # Output layer
    model.add(layers.Dense(1, activation='linear'))

    # Compile
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log', default=1e-3)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='huber',  # Robust to outliers
        metrics=['mae']
    )

    return model


def train_nn_with_tuning(
    training_df: pd.DataFrame,
    max_trials: int = 30,
    epochs_per_trial: int = 50,
    final_epochs: int = 100
) -> Tuple[keras.Model, StandardScaler, LabelEncoder, List[str], Dict]:
    """
    Train neural network with Keras Tuner hyperparameter optimization.
    """
    print("\n" + "=" * 70)
    print("🧠 TRAINING KERAS NEURAL NETWORK WITH HYPERPARAMETER TUNING")
    print("=" * 70)

    # Engineer features
    print("\n📊 Engineering features...")
    training_df = engineer_features(training_df)

    # Filter to players with meaningful minutes
    training_df = training_df[training_df['minutes'] >= 90].copy()
    print(f"   Using {len(training_df)} player-seasons with 90+ minutes")

    # Prepare features
    X, y, scaler, position_encoder, feature_names = prepare_nn_features(training_df, fit_encoders=True)

    print(f"   Features: {X.shape[1]} dimensions")
    print(f"   Target: Points per 90 (mean={y.mean():.2f}, std={y.std():.2f})")

    # Split data
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"\n🔍 Running Keras Tuner ({max_trials} trials, {epochs_per_trial} epochs each)...")
    print("   This may take several minutes...\n")

    # Create tuner
    tuner = kt.BayesianOptimization(
        hypermodel=lambda hp: build_model(hp, input_dim=X.shape[1]),
        objective='val_mae',
        max_trials=max_trials,
        overwrite=True,
        directory='keras_tuner',
        project_name='fpl_nn'
    )

    # Early stopping callback
    early_stop = callbacks.EarlyStopping(
        monitor='val_mae',
        patience=10,
        restore_best_weights=True
    )

    # Search for best hyperparameters
    tuner.search(
        X_train, y_train,
        epochs=epochs_per_trial,
        validation_data=(X_val, y_val),
        callbacks=[early_stop],
        verbose=0
    )

    # Get best hyperparameters
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    best_params = {param: best_hps.get(param) for param in best_hps.values.keys()}

    print("\n✅ Hyperparameter search complete!")
    print("\n📋 Best Hyperparameters:")
    print(f"   • Number of layers: {best_hps.get('n_layers')}")
    print(f"   • Learning rate: {best_hps.get('learning_rate'):.6f}")
    for i in range(best_hps.get('n_layers')):
        print(f"   • Layer {i+1}: {best_hps.get(f'units_{i}')} units, "
              f"dropout={best_hps.get(f'dropout_{i}'):.1f}, "
              f"L2={best_hps.get(f'l2_{i}'):.2e}")

    # Train final model with best hyperparameters
    print(f"\n🏋️ Training final model ({final_epochs} epochs)...")

    final_model = tuner.hypermodel.build(best_hps)

    history = final_model.fit(
        X_train, y_train,
        epochs=final_epochs,
        validation_data=(X_val, y_val),
        callbacks=[
            callbacks.EarlyStopping(monitor='val_mae', patience=15, restore_best_weights=True),
            callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=5, min_lr=1e-6)
        ],
        verbose=0
    )

    # Evaluate
    val_preds = final_model.predict(X_val, verbose=0).flatten()
    val_mae = mean_absolute_error(y_val, val_preds)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))

    print(f"\n📈 Final Model Performance:")
    print(f"   • Validation MAE: {val_mae:.4f} points per 90")
    print(f"   • Validation RMSE: {val_rmse:.4f} points per 90")
    print(f"   • Training stopped at epoch: {len(history.history['loss'])}")

    return final_model, scaler, position_encoder, feature_names, best_params


def train_nn_quick(training_df: pd.DataFrame, epochs: int = 100) -> Tuple:
    """
    Quick training with default architecture (no tuning).
    """
    print("\n🧠 Training Neural Network (quick mode, no tuning)...")

    training_df = engineer_features(training_df)
    training_df = training_df[training_df['minutes'] >= 90].copy()

    X, y, scaler, position_encoder, feature_names = prepare_nn_features(training_df, fit_encoders=True)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # Default architecture
    model = keras.Sequential([
        layers.Input(shape=(X.shape[1],)),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dense(1, activation='linear')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='huber',
        metrics=['mae']
    )

    model.fit(
        X_train, y_train,
        epochs=epochs,
        validation_data=(X_val, y_val),
        callbacks=[
            callbacks.EarlyStopping(monitor='val_mae', patience=15, restore_best_weights=True)
        ],
        verbose=0
    )

    val_preds = model.predict(X_val, verbose=0).flatten()
    print(f"   Validation MAE: {mean_absolute_error(y_val, val_preds):.4f}")

    return model, scaler, position_encoder, feature_names, {}

In [6]:
# ----------------------------------------------------------------------
# 4. PREDICTION & SQUAD MANAGEMENT
# ----------------------------------------------------------------------

def predict_current_season(
    model: keras.Model,
    current_df: pd.DataFrame,
    scaler: StandardScaler,
    position_encoder: LabelEncoder,
    feature_names: List[str]
) -> pd.DataFrame:
    """
    Generate predictions for current season players.
    """
    print("\n🔮 Generating predictions for current season...")

    df = engineer_features(current_df.copy())

    # Prepare features (use fitted scaler/encoder)
    X, _, _, _, _ = prepare_nn_features(
        df,
        fit_encoders=False,
        scaler=scaler,
        position_encoder=position_encoder
    )

    # Predict points per 90
    preds_per_90 = model.predict(X, verbose=0).flatten()

    # Convert to expected GW points (assume ~70-80 mins average)
    # Scale by form and availability
    df['pred_per_90'] = preds_per_90
    df['predicted_points'] = preds_per_90 * 0.85  # Typical match factor

    # Boost for in-form players
    if 'form' in df.columns:
        form_factor = 1 + (df['form'] - df['form'].mean()) / (df['form'].std() + 0.1) * 0.1
        df['predicted_points'] *= form_factor.clip(0.8, 1.3)

    # Penalize doubtful/injured players
    if 'status' in df.columns:
        df.loc[df['status'] == 'd', 'predicted_points'] *= 0.5  # Doubtful
        df.loc[df['status'].isin(['i', 'u', 's', 'n']), 'predicted_points'] *= 0.1  # Out

    df['predicted_points'] = df['predicted_points'].clip(lower=0)

    print(f"   ✅ Generated predictions for {len(df)} players")
    print(f"   Prediction range: {df['predicted_points'].min():.2f} - {df['predicted_points'].max():.2f}")

    return df


def get_user_squad(df: pd.DataFrame) -> List[int]:
    """Get user's current squad via interactive input."""
    print("\n" + "=" * 70)
    print("📋 ENTER YOUR CURRENT SQUAD")
    print("=" * 70)
    print("\nEnter player names (partial match) or IDs. Type 'list' to search.\n")

    squad_ids = []
    pos_counts = {'GKP': 0, 'DEF': 0, 'MID': 0, 'FWD': 0}
    pos_limits = {'GKP': 2, 'DEF': 5, 'MID': 5, 'FWD': 3}

    while len(squad_ids) < 15:
        print(f"\n--- Squad: {len(squad_ids)}/15 | GKP: {pos_counts['GKP']}/2, DEF: {pos_counts['DEF']}/5, MID: {pos_counts['MID']}/5, FWD: {pos_counts['FWD']}/3 ---")

        user_input = input(f"Enter player {len(squad_ids) + 1}: ").strip()

        if user_input.lower() == 'list':
            pos_to_show = input("Position? (GKP/DEF/MID/FWD/ALL): ").strip().upper()
            if pos_to_show == 'ALL':
                print(df[['player_id', 'name', 'team', 'position', 'price']].head(50).to_string())
            elif pos_to_show in pos_limits:
                print(df[df['position'] == pos_to_show][['player_id', 'name', 'team', 'price']].to_string())
            continue

        # Match player
        matched = None
        if user_input.isdigit():
            matches = df[df['player_id'] == int(user_input)]
            if len(matches) == 1:
                matched = matches.iloc[0]

        if matched is None:
            matches = df[df['name'].str.lower().str.contains(user_input.lower(), na=False)]
            if len(matches) == 1:
                matched = matches.iloc[0]
            elif len(matches) > 1:
                print("\n⚠️ Multiple matches:")
                for _, p in matches.head(10).iterrows():
                    print(f"  {p['player_id']}: {p['name']} ({p['team']}, {p['position']})")
                continue
            else:
                print(f"❌ No match for '{user_input}'")
                continue

        pid, pos = matched['player_id'], matched['position']

        if pid in squad_ids:
            print(f"❌ {matched['name']} already in squad")
            continue
        if pos_counts[pos] >= pos_limits[pos]:
            print(f"❌ Already have {pos_limits[pos]} {pos} players")
            continue

        # Team limit check
        current_teams = [df.loc[df['player_id'] == sid, 'team'].iloc[0] for sid in squad_ids]
        if current_teams.count(matched['team']) >= 3:
            print(f"❌ Already have 3 players from {matched['team']}")
            continue

        squad_ids.append(pid)
        pos_counts[pos] += 1
        print(f"✅ Added: {matched['name']} ({matched['team']}, {pos}, £{matched['price']:.1f}m)")

    return squad_ids


def display_squad(squad_ids: List[int], df: pd.DataFrame) -> None:
    """Display squad with predictions."""
    print("\n" + "=" * 70)
    print("📊 YOUR CURRENT SQUAD")
    print("=" * 70)

    squad_df = df[df['player_id'].isin(squad_ids)].copy()

    for pos in ['GKP', 'DEF', 'MID', 'FWD']:
        pos_players = squad_df[squad_df['position'] == pos]
        print(f"\n{pos}:")
        for _, p in pos_players.iterrows():
            print(f"  {p['name']:<20s} {p['team']:<15s} £{p['price']:.1f}m  Pred: {p['predicted_points']:.2f}")

    total_value = squad_df['price'].sum()
    print(f"\n💰 Squad value: £{total_value:.1f}m | Bank: £{100.0 - total_value:.1f}m")
    print(f"📈 Total predicted (best 11): {calculate_best_11_points(squad_ids, df):.2f}")


def calculate_best_11_points(squad_ids: List[int], df: pd.DataFrame) -> float:
    """Calculate best valid starting 11 points."""
    squad_df = df[df['player_id'].isin(squad_ids)].copy()

    gkps = squad_df[squad_df['position'] == 'GKP'].nlargest(1, 'predicted_points')
    defs = squad_df[squad_df['position'] == 'DEF'].nlargest(5, 'predicted_points')
    mids = squad_df[squad_df['position'] == 'MID'].nlargest(5, 'predicted_points')
    fwds = squad_df[squad_df['position'] == 'FWD'].nlargest(3, 'predicted_points')

    selected = []
    selected.extend(defs.nlargest(3, 'predicted_points')['player_id'].tolist())
    selected.extend(mids.nlargest(2, 'predicted_points')['player_id'].tolist())
    selected.extend(fwds.nlargest(1, 'predicted_points')['player_id'].tolist())

    remaining = squad_df[
        (~squad_df['player_id'].isin(selected)) &
        (squad_df['position'] != 'GKP')
    ].nlargest(4, 'predicted_points')
    selected.extend(remaining['player_id'].tolist())

    pts = gkps['predicted_points'].sum() + squad_df[squad_df['player_id'].isin(selected)]['predicted_points'].sum()

    all_starting = pd.concat([gkps, squad_df[squad_df['player_id'].isin(selected)]])
    captain_bonus = all_starting['predicted_points'].max()

    return pts + captain_bonus


def use_sample_squad(df: pd.DataFrame) -> List[int]:
    """Sample squad for testing."""
    print("\n📋 Using sample squad...")
    squad = []
    squad.extend(df[df['position'] == 'GKP'].nsmallest(2, 'price')['player_id'].tolist())
    squad.extend(df[df['position'] == 'DEF'].nsmallest(5, 'price')['player_id'].tolist())
    squad.extend(df[df['position'] == 'MID'].nsmallest(5, 'price')['player_id'].tolist())
    squad.extend(df[df['position'] == 'FWD'].nsmallest(3, 'price')['player_id'].tolist())
    return squad

In [7]:
# ----------------------------------------------------------------------
# 5. TRANSFER OPTIMIZATION (MILP)
# ----------------------------------------------------------------------

def recommend_transfers(
    current_squad_ids: List[int],
    df: pd.DataFrame,
    num_transfers: int,
    budget: float = 100.0
) -> Dict[str, Any]:
    """Recommend optimal transfers using MILP."""
    print(f"\n🔄 Optimizing {num_transfers} transfer(s)...")

    current_squad_df = df[df['player_id'].isin(current_squad_ids)]
    current_value = current_squad_df['price'].sum()
    bank = budget - current_value

    player_ids = df['player_id'].tolist()
    price = dict(zip(player_ids, df['price']))
    position = dict(zip(player_ids, df['position']))
    team = dict(zip(player_ids, df['team']))
    pred_pts = dict(zip(player_ids, df['predicted_points']))

    prob = pulp.LpProblem("FPL_Transfer", pulp.LpMaximize)

    new_squad = pulp.LpVariable.dicts("new_squad", player_ids, cat="Binary")
    transfer_out = pulp.LpVariable.dicts("transfer_out", player_ids, cat="Binary")
    transfer_in = pulp.LpVariable.dicts("transfer_in", player_ids, cat="Binary")
    start = pulp.LpVariable.dicts("start", player_ids, cat="Binary")
    captain = pulp.LpVariable.dicts("captain", player_ids, cat="Binary")

    prob += pulp.lpSum(pred_pts[i] * (start[i] + captain[i]) for i in player_ids)

    for i in player_ids:
        if i in current_squad_ids:
            prob += new_squad[i] == 1 - transfer_out[i]
        else:
            prob += new_squad[i] == transfer_in[i]

    prob += pulp.lpSum(transfer_out[i] for i in player_ids) == num_transfers
    prob += pulp.lpSum(transfer_in[i] for i in player_ids) == num_transfers
    prob += pulp.lpSum(new_squad[i] for i in player_ids) == 15
    prob += pulp.lpSum(price[i] * new_squad[i] for i in player_ids) <= current_value + bank

    for pos, limit in {'GKP': 2, 'DEF': 5, 'MID': 5, 'FWD': 3}.items():
        prob += pulp.lpSum(new_squad[i] for i in player_ids if position[i] == pos) == limit

    for tm in df['team'].unique():
        prob += pulp.lpSum(new_squad[i] for i in player_ids if team[i] == tm) <= 3

    prob += pulp.lpSum(start[i] for i in player_ids) == 11
    for i in player_ids:
        prob += start[i] <= new_squad[i]

    prob += pulp.lpSum(start[i] for i in player_ids if position[i] == 'GKP') == 1
    prob += pulp.lpSum(start[i] for i in player_ids if position[i] == 'DEF') >= 3
    prob += pulp.lpSum(start[i] for i in player_ids if position[i] == 'DEF') <= 5
    prob += pulp.lpSum(start[i] for i in player_ids if position[i] == 'MID') >= 2
    prob += pulp.lpSum(start[i] for i in player_ids if position[i] == 'MID') <= 5
    prob += pulp.lpSum(start[i] for i in player_ids if position[i] == 'FWD') >= 1
    prob += pulp.lpSum(start[i] for i in player_ids if position[i] == 'FWD') <= 3

    prob += pulp.lpSum(captain[i] for i in player_ids) == 1
    for i in player_ids:
        prob += captain[i] <= start[i]

    solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=120)
    prob.solve(solver)

    transfers_out = [i for i in player_ids if pulp.value(transfer_out[i]) > 0.5]
    transfers_in = [i for i in player_ids if pulp.value(transfer_in[i]) > 0.5]
    new_squad_ids = [i for i in player_ids if pulp.value(new_squad[i]) > 0.5]
    starting_ids = [i for i in player_ids if pulp.value(start[i]) > 0.5]
    captain_id = next(i for i in player_ids if pulp.value(captain[i]) > 0.5)

    old_points = calculate_best_11_points(current_squad_ids, df)
    new_points = sum(pred_pts[i] * (1 + (1 if i == captain_id else 0)) for i in starting_ids)
    new_squad_value = sum(price[i] for i in new_squad_ids)

    return {
        'transfers_out': transfers_out,
        'transfers_in': transfers_in,
        'new_squad_ids': new_squad_ids,
        'starting_ids': starting_ids,
        'captain_id': captain_id,
        'old_points': old_points,
        'new_points': new_points,
        'improvement': new_points - old_points,
        'new_squad_value': new_squad_value,
        'new_bank': budget - new_squad_value,
    }


def display_transfers(result: Dict[str, Any], df: pd.DataFrame) -> None:
    """Display transfer recommendations."""
    print("\n" + "=" * 70)
    print("🎯 RECOMMENDED TRANSFERS")
    print("=" * 70)

    print("\n📤 TRANSFER OUT:")
    for pid in result['transfers_out']:
        p = df[df['player_id'] == pid].iloc[0]
        print(f"  ❌ {p['name']:<20s} ({p['team']}, {p['position']}) £{p['price']:.1f}m | {p['predicted_points']:.2f} pts")

    print("\n📥 TRANSFER IN:")
    for pid in result['transfers_in']:
        p = df[df['player_id'] == pid].iloc[0]
        print(f"  ✅ {p['name']:<20s} ({p['team']}, {p['position']}) £{p['price']:.1f}m | {p['predicted_points']:.2f} pts")

    print("\n" + "-" * 70)
    print(f"📊 Old points: {result['old_points']:.2f} → New: {result['new_points']:.2f} (+{result['improvement']:.2f})")
    print(f"💰 Squad value: £{result['new_squad_value']:.1f}m | Bank: £{result['new_bank']:.1f}m")

    cap = df[df['player_id'] == result['captain_id']].iloc[0]
    print(f"\n👑 CAPTAIN: {cap['name']} ({cap['team']}) - {cap['predicted_points']:.2f} pts")

    print("\n--- STARTING 11 ---")
    for pos in ['GKP', 'DEF', 'MID', 'FWD']:
        pos_players = df[(df['player_id'].isin(result['starting_ids'])) & (df['position'] == pos)]
        for _, p in pos_players.iterrows():
            marks = (" (C)" if p['player_id'] == result['captain_id'] else "") + \
                    (" 🆕" if p['player_id'] in result['transfers_in'] else "")
            print(f"  {pos:3s} {p['name']:<20s} £{p['price']:.1f}m  {p['predicted_points']:.2f} pts{marks}")

In [8]:
# ----------------------------------------------------------------------
# 6. MAIN EXECUTION - DATA LOADING
# ----------------------------------------------------------------------

print("=" * 70)
print("⚽ FPL TRANSFER OPTIMIZER - Keras Neural Network (3 Seasons)")
print("=" * 70)

# Fetch all seasons data
training_df, current_df = fetch_all_seasons_data(verify_ssl=False)

print(f"\n✅ Data loaded successfully!")

⚽ FPL TRANSFER OPTIMIZER - Keras Neural Network (3 Seasons)

📊 FETCHING MULTI-SEASON FPL DATA

📡 Fetching current season (2024-25) from FPL API...
   ✅ Loaded 759 players from current season
📡 Fetching historical season (2023-24) from GitHub...
   ✅ Loaded 865 players from 2023-24
📡 Fetching historical season (2022-23) from GitHub...
   ✅ Loaded 778 players from 2022-23

📈 Total dataset: 2402 player-seasons
   Seasons loaded: 3
   • 2024-25: 759 players
   • 2023-24: 865 players
   • 2022-23: 778 players

✅ Data loaded successfully!


In [9]:
# ----------------------------------------------------------------------
# 7. ENTER YOUR SQUAD
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
choice = input("Enter squad manually (M) or use sample (S)? [M/S]: ").strip().upper()

if choice == 'M':
    current_squad = get_user_squad(current_df)
else:
    current_squad = use_sample_squad(current_df)

print(f"\n✅ Squad of {len(current_squad)} players selected.")


Enter squad manually (M) or use sample (S)? [M/S]: S

📋 Using sample squad...

✅ Squad of 15 players selected.


In [10]:
# ----------------------------------------------------------------------
# 8. TRAIN NEURAL NETWORK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("🧠 NEURAL NETWORK TRAINING")
print("=" * 70)

tuning_choice = input("\nFull hyperparameter tuning (F) or quick mode (Q)? [F/Q]: ").strip().upper()

if tuning_choice == 'Q':
    model, scaler, position_encoder, feature_names, best_params = train_nn_quick(training_df)
else:
    print("\n⚙️ Tuning Configuration:")
    try:
        max_trials = int(input("   Keras Tuner trials [10-100, default=30]: ").strip() or "30")
        max_trials = max(10, min(100, max_trials))
    except ValueError:
        max_trials = 30

    model, scaler, position_encoder, feature_names, best_params = train_nn_with_tuning(
        training_df,
        max_trials=max_trials
    )


🧠 NEURAL NETWORK TRAINING

⚙️ Tuning Configuration:

🧠 TRAINING KERAS NEURAL NETWORK WITH HYPERPARAMETER TUNING

📊 Engineering features...
   Using 1367 player-seasons with 90+ minutes
   Features: 34 dimensions
   Target: Points per 90 (mean=4.04, std=1.77)

🔍 Running Keras Tuner (10 trials, 50 epochs each)...
   This may take several minutes...


✅ Hyperparameter search complete!

📋 Best Hyperparameters:
   • Number of layers: 2
   • Learning rate: 0.005382
   • Layer 1: 128 units, dropout=0.0, L2=3.56e-05
   • Layer 2: 64 units, dropout=0.0, L2=1.13e-06

🏋️ Training final model (100 epochs)...

📈 Final Model Performance:
   • Validation MAE: 0.2553 points per 90
   • Validation RMSE: 0.4773 points per 90
   • Training stopped at epoch: 67


In [11]:
# ----------------------------------------------------------------------
# 9. GENERATE PREDICTIONS
# ----------------------------------------------------------------------

current_df = predict_current_season(model, current_df, scaler, position_encoder, feature_names)

# Display squad with predictions
display_squad(current_squad, current_df)


🔮 Generating predictions for current season...
   ✅ Generated predictions for 759 players
   Prediction range: 0.10 - 68.74

📊 YOUR CURRENT SQUAD

GKP:
  Setford              Arsenal         £3.9m  Pred: 1.01
  Wright               Aston Villa     £3.9m  Pred: 1.01

DEF:
  Clarke               Arsenal         £3.8m  Pred: 1.63
  Nichols              Arsenal         £3.8m  Pred: 1.63
  Delcroix             Burnley         £3.8m  Pred: 1.63
  Lucas Pires          Burnley         £3.8m  Pred: 0.11
  J.Araujo             Bournemouth     £3.8m  Pred: 1.63

MID:
  Dowman               Arsenal         £4.3m  Pred: 13.87
  Konak                Brentford       £4.3m  Pred: 1.66
  D.Essugo             Chelsea         £4.3m  Pred: 0.17
  Devenny              Crystal Palace  £4.3m  Pred: 4.70
  Reed                 Fulham          £4.3m  Pred: 61.98

FWD:
  Barnes               Burnley         £4.3m  Pred: 7.85
  Marc Guiu            Chelsea         £4.2m  Pred: 3.89
  Danns                Liverp

In [12]:
# ----------------------------------------------------------------------
# 10. GET NUMBER OF TRANSFERS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("🔄 TRANSFER OPTIMIZATION")
print("=" * 70)

while True:
    try:
        num_transfers = int(input("\nHow many transfers do you want to make? [1-15]: ").strip())
        if 1 <= num_transfers <= 15:
            break
        print("❌ Enter a number between 1 and 15.")
    except ValueError:
        print("❌ Enter a valid number.")

print(f"\n✅ Optimizing {num_transfers} transfer(s)...")


🔄 TRANSFER OPTIMIZATION

How many transfers do you want to make? [1-15]: 5

✅ Optimizing 5 transfer(s)...


In [13]:
# ----------------------------------------------------------------------
# 11. OPTIMIZE AND DISPLAY TRANSFERS
# ----------------------------------------------------------------------

result = recommend_transfers(current_squad, current_df, num_transfers=num_transfers)
display_transfers(result, current_df)


🔄 Optimizing 5 transfer(s)...

🎯 RECOMMENDED TRANSFERS

📤 TRANSFER OUT:
  ❌ Clarke               (Arsenal, DEF) £3.8m | 1.63 pts
  ❌ J.Araujo             (Bournemouth, DEF) £3.8m | 1.63 pts
  ❌ Konak                (Brentford, MID) £4.3m | 1.66 pts
  ❌ D.Essugo             (Chelsea, MID) £4.3m | 0.17 pts
  ❌ Devenny              (Crystal Palace, MID) £4.3m | 4.70 pts

📥 TRANSFER IN:
  ✅ Knight               (Brighton, MID) £4.5m | 61.87 pts
  ✅ Oriola               (Brighton, MID) £4.5m | 61.87 pts
  ✅ Sosa                 (Crystal Palace, DEF) £4.8m | 30.95 pts
  ✅ Bornauw              (Leeds, DEF) £3.8m | 62.70 pts
  ✅ Neil                 (Sunderland, MID) £4.8m | 68.74 pts

----------------------------------------------------------------------
📊 Old points: 163.45 → New: 445.12 (+281.67)
💰 Squad value: £63.0m | Bank: £37.0m

👑 CAPTAIN: Neil (Sunderland) - 68.74 pts

--- STARTING 11 ---
  GKP Setford              £3.9m  1.01 pts
  DEF Delcroix             £3.8m  1.63 pts
  DEF Sosa

In [14]:
# ----------------------------------------------------------------------
# 12. ADDITIONAL INSIGHTS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("💡 ADDITIONAL INSIGHTS")
print("=" * 70)

# Top scorers not in squad
not_in_squad = current_df[~current_df['player_id'].isin(result['new_squad_ids'])]
top_missed = not_in_squad.nlargest(5, 'predicted_points')

print("\n🔥 Top 5 predicted scorers NOT in your squad:")
for _, p in top_missed.iterrows():
    print(f"  • {p['name']} ({p['team']}, {p['position']}) - £{p['price']:.1f}m - {p['predicted_points']:.2f} pts")

# Value picks
current_df['value'] = current_df['predicted_points'] / current_df['price']
best_value = current_df.nlargest(5, 'value')
print("\n💎 Top 5 value picks (pts per £m):")
for _, p in best_value.iterrows():
    mark = "✓" if p['player_id'] in result['new_squad_ids'] else " "
    print(f"  {mark} {p['name']} ({p['team']}, {p['position']}) - {p['value']:.2f} pts/£m")

# Differentials
if 'selected_by_percent' in current_df.columns:
    differentials = current_df[current_df['selected_by_percent'] < 5].nlargest(5, 'predicted_points')
    print("\n🎲 Top differentials (<5% ownership):")
    for _, p in differentials.iterrows():
        mark = "✓" if p['player_id'] in result['new_squad_ids'] else " "
        print(f"  {mark} {p['name']} ({p['team']}) - {p['selected_by_percent']:.1f}% - {p['predicted_points']:.2f} pts")

print("\n" + "=" * 70)
print("Good luck with your transfers! ⚽🚀")
print("=" * 70)


💡 ADDITIONAL INSIGHTS

🔥 Top 5 predicted scorers NOT in your squad:
  • Kostoulas (Brighton, FWD) - £4.8m - 24.36 pts
  • Clyne (Crystal Palace, DEF) - £3.8m - 21.54 pts
  • Enes Ünal (Bournemouth, FWD) - £5.4m - 21.46 pts
  • Uche (Crystal Palace, FWD) - £5.5m - 17.89 pts
  • Ngumoha (Liverpool, MID) - £4.4m - 17.23 pts

💎 Top 5 value picks (pts per £m):
  ✓ Bornauw (Leeds, DEF) - 16.50 pts/£m
  ✓ Reed (Fulham, MID) - 14.41 pts/£m
  ✓ Neil (Sunderland, MID) - 14.32 pts/£m
  ✓ Knight (Brighton, MID) - 13.75 pts/£m
  ✓ Oriola (Brighton, MID) - 13.75 pts/£m

🎲 Top differentials (<5% ownership):
  ✓ Neil (Sunderland) - 0.0% - 68.74 pts
  ✓ Bornauw (Leeds) - 0.1% - 62.70 pts
  ✓ Reed (Fulham) - 0.3% - 61.98 pts
  ✓ Knight (Brighton) - 0.1% - 61.87 pts
  ✓ Oriola (Brighton) - 0.0% - 61.87 pts

Good luck with your transfers! ⚽🚀


In [15]:
# ----------------------------------------------------------------------
# 13. OPTIONAL: Model Architecture Summary
# ----------------------------------------------------------------------

print("\n📐 Neural Network Architecture:")
model.summary()


📐 Neural Network Architecture:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 128)            │         4,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,405 (150.02 KB)

 Trainable params: 12,801 (50.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 25,604 (100.02 KB)

In [16]:
# ----------------------------------------------------------------------
# 14. OPTIONAL: Run another scenario
# ----------------------------------------------------------------------

def run_another():
    global result, current_squad

    use_new = input("\nUse new squad (N) or original (O)? [N/O]: ").strip().upper()
    squad = result['new_squad_ids'] if use_new == 'N' else current_squad

    while True:
        try:
            n = int(input("How many transfers? [1-15]: ").strip())
            if 1 <= n <= 15:
                break
        except ValueError:
            pass

    new_result = recommend_transfers(squad, current_df, num_transfers=n)
    display_transfers(new_result, current_df)
    return new_result

# Uncomment to run:
result = run_another()


Use new squad (N) or original (O)? [N/O]: O
How many transfers? [1-15]: 5

🔄 Optimizing 5 transfer(s)...

🎯 RECOMMENDED TRANSFERS

📤 TRANSFER OUT:
  ❌ Clarke               (Arsenal, DEF) £3.8m | 1.63 pts
  ❌ J.Araujo             (Bournemouth, DEF) £3.8m | 1.63 pts
  ❌ Konak                (Brentford, MID) £4.3m | 1.66 pts
  ❌ D.Essugo             (Chelsea, MID) £4.3m | 0.17 pts
  ❌ Devenny              (Crystal Palace, MID) £4.3m | 4.70 pts

📥 TRANSFER IN:
  ✅ Knight               (Brighton, MID) £4.5m | 61.87 pts
  ✅ Oriola               (Brighton, MID) £4.5m | 61.87 pts
  ✅ Sosa                 (Crystal Palace, DEF) £4.8m | 30.95 pts
  ✅ Bornauw              (Leeds, DEF) £3.8m | 62.70 pts
  ✅ Neil                 (Sunderland, MID) £4.8m | 68.74 pts

----------------------------------------------------------------------
📊 Old points: 163.45 → New: 445.12 (+281.67)
💰 Squad value: £63.0m | Bank: £37.0m

👑 CAPTAIN: Neil (Sunderland) - 68.74 pts

--- STARTING 11 ---
  GKP Setford        

In [ ]:
# ----------------------------------------------------------------------
# 15. OPTIONAL: Save model
# ----------------------------------------------------------------------

# Uncomment to save:
# model.save('fpl_nn_model.keras')
# print("✅ Model saved to 'fpl_nn_model.keras'")